# nb_05 — Interactive scatter + light-curve explorer

Goal (see `docs/SPEC_V01.md`, rough plan step 6): a reusable function — click a point in any
scatter plot (CMD, period-amplitude, ...), see that object's light curve on the right, with
toggles to fold on a period and to show/hide flux errors. Implemented once in
`src/visualization/lc_explorer.py` (`interactive_scatter_lc`), not copy-pasted per notebook —
this notebook is a demo of it against nb_03's and nb_04's actual outputs.

**Needs a live Jupyter kernel with widget support** (built on `plotly.graph_objects.FigureWidget`
+ `ipywidgets`) — the click-to-update behavior can't be exercised by non-interactive execution
(this notebook was run with `jupyter nbconvert --execute`, which builds the widgets and confirms
they don't error, but can't simulate an actual mouse click). Try clicking points for real in the
RSP JupyterLab session; this hasn't been checked in VSCode or other IDEs.

**Environment note:** plotly's `FigureWidget` (as of plotly 6.x) requires the `anywidget`
package, which wasn't in `pyproject.toml`'s pinned list and had to be installed separately
(`pip install --user anywidget`) — see the updated `README.md`.

**Housekeeping note:** re-running this notebook and saving from a live Jupyter session embeds a
full widget-state snapshot in `metadata.widgets` — in this case ~21 MB (the scatter data
serialized into the saved model state), vs. ~11 KB without it. That snapshot only exists to let
static viewers (nbviewer, GitHub) show a frozen widget preview, which isn't useful here anyway
since the whole point is clicking it live — so it's stripped from the committed version
(`nb["metadata"].pop("widgets", None)` after execution). Strip it again before committing if you
re-run this and save from Jupyter directly.


In [1]:
import sys
from pathlib import Path

# src/ isn't pip-installed on RSP (pyproject.toml is pinned to Python 3.14, newer than RSP's
# 3.13.9 kernel — see README) — import it straight from the repo checkout instead.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import lsdb
from datapaths import Datapaths
from visualization import interactive_scatter_lc

dp = Datapaths()
periods_cat = lsdb.open_catalog(dp["dp2_subset"] / "dia_object_lc_10plus_with_periods")
print(sorted(periods_cat.columns))


['best_period_days', 'best_period_power', 'dec', 'diaObjectForcedSource', 'diaObjectId', 'diaSource', 'duration_days', 'field', 'g_amp_p90p10', 'g_mag_median', 'g_period_days', 'g_period_power', 'i_amp_p90p10', 'i_mag_median', 'i_period_days', 'i_period_power', 'max_reliability', 'median_cadence_gap_days', 'multiband_period_days', 'multiband_period_power', 'nDiaSources', 'r_amp_p90p10', 'r_mag_median', 'r_period_days', 'r_period_power', 'ra', 'tract', 'u_amp_p90p10', 'u_mag_median', 'u_period_days', 'u_period_power', 'y_amp_p90p10', 'y_mag_median', 'y_period_days', 'y_period_power', 'z_amp_p90p10', 'z_mag_median', 'z_period_days', 'z_period_power']


## 1. Load once

`interactive_scatter_lc`'s `lc_df` needs to be already materialized (fetching a light curve per
click has to be fast — no lazy per-click `.compute()`). This subset is small enough (7036
objects, ~90 MB) to just pull the whole thing once and reuse it as both the light-curve source
and the source for the scatter plots below.


In [2]:
cols = [
    "diaObjectId", "field", "diaSource",
    "g_mag_median", "r_mag_median", "i_mag_median",
    "r_amp_p90p10", "best_period_days", "best_period_power",
    "multiband_period_days", "multiband_period_power","diaObjectForcedSource"
]
full_df = periods_cat[cols].compute()
print(full_df.shape)


Computing Catalog:   0%|          | 0/8 [00:00<?, ?it/s]

(7036, 12)


## 2. Demo: color-magnitude diagram (nb_03)

`g-r` vs `r`, colored by field, folded on `best_period_days` (nb_04's single-band period) when
available. Click a point on the left to load its light curve on the right; toggle folded vs.
unfolded and flux errors on/off.


In [3]:
full_df['diaObjectForcedSource'].columns

['band',
 'coord_dec',
 'coord_ra',
 'diff_PixelFlags_nodataCenter',
 'invalidPsfFlag',
 'midpointMjdTai',
 'pixelFlags_bad',
 'pixelFlags_cr',
 'pixelFlags_crCenter',
 'pixelFlags_edge',
 'pixelFlags_interpolated',
 'pixelFlags_interpolatedCenter',
 'pixelFlags_nodata',
 'pixelFlags_saturated',
 'pixelFlags_saturatedCenter',
 'pixelFlags_suspect',
 'pixelFlags_suspectCenter',
 'psfDiffFlux',
 'psfDiffFlux_flag',
 'psfDiffFluxErr',
 'psfFlux',
 'psfFlux_flag',
 'psfFluxErr',
 'psfMag',
 'psfMagErr',
 'visit']

In [4]:
cmd_df = full_df.assign(gr=full_df["g_mag_median"] - full_df["r_mag_median"]).dropna(subset=["gr", "r_mag_median"])
print(cmd_df.shape)

interactive_scatter_lc(
    scatter_df=cmd_df,
    x_col="gr",
    y_col="r_mag_median",
    lc_df=full_df,
    color_col="field",
    period_col="best_period_days",
    scatter_title="CMD: g-r vs r",
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource"
)


(6907, 13)


    'data': [{'hoverinfo': 'text',
              'marker': {'color': [#636EFA, #…

## 3. Demo: period-amplitude diagram (nb_04 x nb_02)

`multiband_period_days` (log-scaled, nb_04's higher-coverage period) vs. `r_amp_p90p10`
(nb_02's robust amplitude), folded on the same `multiband_period_days`. This is exactly the
combination the spec's rough plan step 5 asks for as a static plot — here it's interactive
instead, in the same function used for the CMD above.


In [5]:
pa_df = full_df.dropna(subset=["multiband_period_days", "r_amp_p90p10"])
print(pa_df.shape)

interactive_scatter_lc(
    scatter_df=pa_df,
    x_col="multiband_period_days",
    y_col="r_amp_p90p10",
    lc_df=full_df,
    color_col="field",
    period_col="multiband_period_days",
    scatter_title="period-amplitude: multiband period vs r-band amplitude",
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource",
    x_log=True,
)


(4493, 12)


    'data': [{'hoverinfo': 'text',
              'marker': {'color': [#636EFA, #…

## Next

`src/visualization/lc_explorer.py`'s `interactive_scatter_lc` is the reusable piece; this
notebook is just two example calls against real outputs. Open questions, not resolved here:

- **Not tested outside JupyterLab on RSP.** The spec explicitly flags Jupyter-vs-VSCode/IDE
  differences for interactive widgets — unverified either way here.
- **No legend for a categorical `color_col`.** A single `go.Scatter` trace can't carry a
  per-category legend the way separate traces per category would — the category rides along in
  the hover text instead (see `field=...` when hovering above), which is a real but weaker
  substitute.
- **`lc_df` must already be materialized** — no support for handing it a lazy `lsdb.Catalog`
  and computing per click. Fine at this subset's scale (7036 objects); would need rethinking
  before pointing this at anything close to the full collection.
- **Doesn't add its own quality filtering.** All of nb_04's caveats about `*_period_power` not
  being a calibrated false-alarm probability still apply when browsing by period here — clicking
  a high-power point doesn't mean the fold is real.
- **Single light-curve panel, not one subplot per band** — matches the spec's wording ("the LC
  plotting panel", singular), but multi-band light curves with very different flux scales can
  be hard to read overlaid; worth revisiting if that turns out to matter in practice.
